# Google Flight Search: PK to UK after 2 Sep 2026

Is the PK to UK booking collapse after 2 Sep 2026 a **price, visibility or demand change on Google Flights**,
independent of paid search? Source queries: `gfs_pk_uk.sql` (same folder).

- Queries run with the **billing** project `gcp-stg-prj-data-dgm-ds01`; tables live in the **data** project
  `gcp-prd-prj-data-dgm01` (see `results/data_sources/GFS_QUERY_GUIDE.md`).
- Weighted means only, weeks start on Sunday, both years so September 2025 is the control.
- Every result is saved to `OUT_DIR` as CSV, plus one Excel workbook at the end.

Run top to bottom. Change the dates or airport lists in the settings cell only.

In [ ]:
# !pip install -q google-cloud-bigquery google-cloud-bigquery-storage db-dtypes pandas matplotlib openpyxl
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from google.cloud import bigquery

pd.set_option("display.width", 200, "display.max_columns", 30)

## 1. Settings

In [ ]:
# Two projects: data (in table names) vs billing (runs and pays for the queries). Never merge them.
GCP_PROJECT     = "gcp-prd-prj-data-dgm01"     # data project (read from)
GCP_PROJECT_CLI = "gcp-stg-prj-data-dgm-ds01"  # billing / client project

GFS_DS   = f"{GCP_PROJECT}.bq_lnd_google_flight_search"
TBL_COMP = f"`{GFS_DS}.lnd_gfs_competitiveness`"
TBL_SHOP = f"`{GFS_DS}.lnd_gfs_shopping_competitiveness`"

PK_ORIGINS = ["ISB", "LHE", "KHI", "MUX", "SKT", "PEW", "LYP", "UET"]
UK_IE      = ["LHR", "LGW", "MAN", "BHX", "EDI", "GLA", "STN", "LTN", "BRS", "NCL", "LBA", "DUB", "ORK", "SNN"]

SWITCH     = "2026-09-02"   # VBB switch in PK
END        = "2026-10-05"   # last day to read
TREND_FROM = "2025-06-01"   # weekly trend start (covers Sep 2025 as the control)
COMP_FROM  = "2026-06-01"   # competitor and partner checks start
ROUTE_FROM = "08-01"        # route table: 1 Aug to END, this year and last year

USER_COUNTRY = None        # set after the q0b check, e.g. "PK", to keep only Google Flights users in Pakistan
                           # (the paid-search audience). None = all user countries.

MAX_GB = 50                 # refuse any query that would scan more than this (dry run first)

OUT_DIR = Path("gfs_pk_uk_out"); OUT_DIR.mkdir(exist_ok=True)

sql_list = lambda xs: ", ".join(f"'{x}'" for x in xs)
PK, UK = sql_list(PK_ORIGINS), sql_list(UK_IE)
UC = f"AND user_country = '{USER_COUNTRY}'" if USER_COUNTRY else ""   # competitiveness table only
END_LY = str(pd.Timestamp(END) - pd.DateOffset(years=1))[:10]
print("PK origins:", PK); print("UK+IE:", UK); print("Last year control ends", END_LY); print("User country filter:", USER_COUNTRY or "none")

## 2. BigQuery client (billing project)

In [ ]:
client = bigquery.Client(project=GCP_PROJECT_CLI)

def bq(sql: str, name: str) -> pd.DataFrame:
    """Dry-run (size check), run, save to OUT_DIR/<name>.csv and return the DataFrame."""
    dry = client.query(sql, job_config=bigquery.QueryJobConfig(dry_run=True, use_query_cache=False))
    gb = dry.total_bytes_processed / 1e9
    if gb > MAX_GB:
        raise RuntimeError(f"{name}: would scan {gb:.1f} GB (> MAX_GB={MAX_GB}). Narrow the dates or raise MAX_GB.")
    df = client.query(sql).to_dataframe(create_bqstorage_client=True)
    df.to_csv(OUT_DIR / f"{name}.csv", index=False)
    print(f"{name}: {len(df):,} rows, {gb:.2f} GB scanned -> {OUT_DIR / (name + '.csv')}")
    return df

print("billing project:", GCP_PROJECT_CLI, "| data project:", GCP_PROJECT)

## 3. Checks: date coverage, `user_country` format, partners on PK to UK routes

Read these before the rest: if the last day is before `END`, the latest weeks are partial. If only `QR` shows up
in the partner list, query 4 (competitors) has nothing to show.

In [ ]:
q0a = f"""
SELECT MIN(date) AS first_day, MAX(date) AS last_day
FROM {TBL_COMP}
WHERE partner = 'QR'
"""
q0b = f"""
SELECT user_country, SUM(price_bucket_lowest_weight) AS w
FROM {TBL_COMP}
WHERE partner = 'QR' AND date BETWEEN '2026-08-01' AND '{END}'
  AND origin_airport_code IN ({PK})
GROUP BY user_country ORDER BY w DESC LIMIT 20
"""
q0c = f"""
SELECT partner, COUNT(*) AS rows_, SUM(participation_rate_weight) AS w
FROM {TBL_COMP}
WHERE date BETWEEN '{COMP_FROM}' AND '{END}'
  AND origin_airport_code IN ({PK})
  AND destination_airport_code IN ({UK})
GROUP BY partner ORDER BY w DESC
"""
q0d = f"""
SELECT partner, COUNT(*) AS rows_, SUM(participation_weight) AS w
FROM {TBL_SHOP}
WHERE date BETWEEN '{COMP_FROM}' AND '{END}'
  AND origin IN ({PK})
  AND destination IN ({UK})
GROUP BY partner ORDER BY w DESC
"""
q0e = f"""
SELECT table_name, column_name, data_type
FROM `{GFS_DS}.INFORMATION_SCHEMA.COLUMNS`
WHERE table_name IN ('lnd_gfs_competitiveness', 'lnd_gfs_shopping_competitiveness')
ORDER BY table_name, ordinal_position
"""
schema    = bq(q0e, "q0e_schema");        display(schema)
coverage  = bq(q0a, "q0a_coverage");      display(coverage)
countries = bq(q0b, "q0b_user_country");  display(countries)
partners  = bq(q0c, "q0c_partners");      display(partners)
partners_shop = bq(q0d, "q0d_partners_shopping"); display(partners_shop)

last_day = pd.Timestamp(coverage["last_day"].iloc[0])
if last_day < pd.Timestamp(END):
    print(f"WARNING: data ends {last_day.date()}, before END={END}.")
print("Weekly volumes below are per day (weight / days with data), so partial weeks do not look like drops.")
if set(partners_shop["partner"]) <= {"QR"}:
    print("Only QR rows on PK-UK routes: query 4 (competitors) will show QR only.")

## 4. QR price competitiveness and visibility, PK origins, weekly, UK+IE vs other destinations

In [ ]:
q1 = f"""
SELECT
  DATE_TRUNC(date, WEEK(SUNDAY)) AS week,
  IF(destination_airport_code IN ({UK}), 'UK+IE', 'other') AS dest_group,
  SAFE_DIVIDE(SUM(price_bucket_lowest * price_bucket_lowest_weight), SUM(price_bucket_lowest_weight)) AS pct_cheapest,
  SAFE_DIVIDE(SUM(price_bucket_not_lowest * price_bucket_not_lowest_weight), SUM(price_bucket_not_lowest_weight)) AS pct_not_cheapest,
  SAFE_DIVIDE(SUM(average_positive_price_diff * average_positive_price_diff_weight),
              SUM(average_positive_price_diff_weight)) AS avg_gap_when_more_expensive,
  SAFE_DIVIDE(SUM(participation_rate * participation_rate_weight), SUM(participation_rate_weight)) AS participation,
  SAFE_DIVIDE(SUM(top_impression_share * top_impression_share_weight), SUM(top_impression_share_weight)) AS top_impr_share,
  SUM(price_bucket_lowest_weight) AS demand_weight,
  COUNT(DISTINCT date) AS days
FROM {TBL_COMP}
WHERE partner = 'QR'
  AND date BETWEEN '{TREND_FROM}' AND '{END}'
  AND origin_airport_code IN ({PK})
  {UC}
GROUP BY week, dest_group
ORDER BY dest_group, week
"""
w1 = bq(q1, "q1_competitiveness_weekly")
w1["week"] = pd.to_datetime(w1["week"])
w1["demand_per_day"] = w1["demand_weight"] / w1["days"]
w1.tail(10)

### Charts

Each panel is one metric: PK to UK+IE (blue) against PK to other destinations (grey, the comparison).
The dashed line is the 2 Sep 2026 switch. The second figure overlays 2026 on the same weeks of 2025 for UK+IE.

In [ ]:
BLUE, GREY = "#4A86E8", "#8C8C8C"
SW = pd.Timestamp(SWITCH)

def panels(df, metrics, title, fname):
    fig, axes = plt.subplots(2, (len(metrics) + 1) // 2, figsize=(14, 6.5), sharex=True)
    for ax, (col, label) in zip(axes.flat, metrics.items()):
        for grp, colr in (("other", GREY), ("UK+IE", BLUE)):
            d = df[df["dest_group"] == grp]
            ax.plot(d["week"], d[col], color=colr, lw=2, label=f"PK to {grp}")
        ax.axvline(SW, color="#333333", ls="--", lw=1)
        ax.set_title(label, fontsize=10, loc="left")
        ax.grid(axis="y", color="#E5E5E5"); ax.spines[["top", "right"]].set_visible(False)
        ax.xaxis.set_major_locator(mdates.MonthLocator(interval=3))
        ax.xaxis.set_major_formatter(mdates.DateFormatter("%b %y"))
    for ax in list(axes.flat)[len(metrics):]:
        ax.set_visible(False)
    axes.flat[0].legend(frameon=False, fontsize=9)
    fig.suptitle(title, x=0.01, ha="left", fontsize=12)
    fig.tight_layout(); fig.savefig(OUT_DIR / fname, dpi=150); plt.show()

panels(w1, {"pct_cheapest": "Share of searches where QR is cheapest",
            "avg_gap_when_more_expensive": "Price gap when QR is more expensive",
            "participation": "Participation (QR shown)",
            "top_impr_share": "Top impression share",
            "demand_per_day": "Demand weight per day (relative volume)"},
       "GFS competitiveness, PK origins, weekly", "c1_competitiveness_weekly.png")

def yoy(df, metrics, title, fname):
    d = df[df["dest_group"] == "UK+IE"].copy()
    d["yr"] = d["week"].dt.year
    d["wk"] = d["week"].dt.isocalendar().week.astype(int)
    d = d[d["week"].dt.month.between(6, 10)]
    fig, axes = plt.subplots(1, len(metrics), figsize=(14, 3.6))
    for ax, (col, label) in zip(axes, metrics.items()):
        for yr, colr in ((2025, GREY), (2026, BLUE)):
            s = d[d["yr"] == yr]
            ax.plot(s["wk"], s[col], color=colr, lw=2, label=str(yr))
        ax.axvline(pd.Timestamp(SWITCH).isocalendar().week, color="#333333", ls="--", lw=1)
        ax.set_title(label, fontsize=10, loc="left"); ax.set_xlabel("ISO week")
        ax.grid(axis="y", color="#E5E5E5"); ax.spines[["top", "right"]].set_visible(False)
    axes[0].legend(frameon=False, fontsize=9)
    fig.suptitle(title, x=0.01, ha="left", fontsize=12)
    fig.tight_layout(); fig.savefig(OUT_DIR / fname, dpi=150); plt.show()

yoy(w1, {"pct_cheapest": "QR cheapest", "participation": "Participation", "demand_per_day": "Demand weight per day"},
    "PK to UK+IE, 2026 vs 2025 (same weeks)", "c2_competitiveness_yoy.png")

## 5. QR shopping behaviour on Google Flights: rank, best flights, selection when shown

In [ ]:
q2 = f"""
SELECT
  DATE_TRUNC(date, WEEK(SUNDAY)) AS week,
  IF(destination IN ({UK}), 'UK+IE', 'other') AS dest_group,
  SAFE_DIVIDE(SUM(avg_rank * avg_rank_weight), SUM(avg_rank_weight)) AS avg_rank,
  SAFE_DIVIDE(SUM(in_best_flights_when_present * in_best_flights_when_present_weight),
              SUM(in_best_flights_when_present_weight)) AS pct_best_flights,
  SAFE_DIVIDE(SUM(participation * participation_weight), SUM(participation_weight)) AS participation,
  SAFE_DIVIDE(SUM(pct_of_all_outbound_selections * pct_of_all_outbound_selections_weight),
              SUM(pct_of_all_outbound_selections_weight)) AS share_of_all_selections,
  SAFE_DIVIDE(SUM(pct_of_outbound_selections_given_partner_was_present
                  * pct_of_outbound_selections_given_partner_was_present_weight),
              SUM(pct_of_outbound_selections_given_partner_was_present_weight)) AS selection_rate_when_present,
  SUM(participation_weight) AS volume_weight,
  COUNT(DISTINCT date) AS days
FROM {TBL_SHOP}
WHERE partner = 'QR'
  AND date BETWEEN '{TREND_FROM}' AND '{END}'
  AND origin IN ({PK})
GROUP BY week, dest_group
ORDER BY dest_group, week
"""
w2 = bq(q2, "q2_shopping_weekly")
w2["week"] = pd.to_datetime(w2["week"])
w2["volume_per_day"] = w2["volume_weight"] / w2["days"]
panels(w2, {"avg_rank": "Average rank (lower is better)",
            "pct_best_flights": "In Best flights when shown",
            "participation": "Participation",
            "share_of_all_selections": "Share of all outbound selections",
            "selection_rate_when_present": "Selection rate when shown",
            "volume_per_day": "Volume weight per day (relative)"},
       "GFS shopping, PK origins, weekly", "c3_shopping_weekly.png")
yoy(w2, {"avg_rank": "Average rank", "selection_rate_when_present": "Selection rate when shown",
         "volume_per_day": "Volume weight per day"},
    "PK to UK+IE shopping, 2026 vs 2025 (same weeks)", "c4_shopping_yoy.png")

## 6. PK to UK routes: before vs from 2 Sep, this year and last year

`change_2026` and `change_2025` are the before-to-after moves within each year; `net` is 2026 minus 2025, so a
move that also happened last September (seasonality) nets out.

In [ ]:
q3 = f"""
SELECT
  EXTRACT(YEAR FROM date) AS yr,
  IF(EXTRACT(DAYOFYEAR FROM date) < EXTRACT(DAYOFYEAR FROM DATE '{SWITCH}'), 'before 2 Sep', 'from 2 Sep') AS half,
  CONCAT(origin_airport_code, '-', destination_airport_code) AS ond,
  SAFE_DIVIDE(SUM(price_bucket_lowest * price_bucket_lowest_weight), SUM(price_bucket_lowest_weight)) AS pct_cheapest,
  SAFE_DIVIDE(SUM(average_positive_price_diff * average_positive_price_diff_weight),
              SUM(average_positive_price_diff_weight)) AS avg_gap_when_more_expensive,
  SAFE_DIVIDE(SUM(participation_rate * participation_rate_weight), SUM(participation_rate_weight)) AS participation,
  SUM(price_bucket_lowest_weight) AS demand_weight,
  COUNT(DISTINCT date) AS days
FROM {TBL_COMP}
WHERE partner = 'QR'
  AND (date BETWEEN '2026-{ROUTE_FROM}' AND '{END}' OR date BETWEEN '2025-{ROUTE_FROM}' AND '{END_LY}')
  AND origin_airport_code IN ({PK})
  AND destination_airport_code IN ({UK})
  {UC}
GROUP BY yr, half, ond
ORDER BY yr, ond, half
"""
r3 = bq(q3, "q3_routes_before_after")

# Demand weight is a total: per day of data, so halves of different length (or a late load) compare.
r3["demand_per_day"] = r3["demand_weight"] / r3["days"]

metrics = ["pct_cheapest", "avg_gap_when_more_expensive", "participation", "demand_per_day"]
wide = r3.pivot_table(index="ond", columns=["yr", "half"], values=metrics)
out = pd.DataFrame(index=wide.index)
for m in metrics:
    for yr in (2025, 2026):
        b, a = wide.get((m, yr, "before 2 Sep")), wide.get((m, yr, "from 2 Sep"))
        if b is None or a is None:
            continue
        out[f"{m}_{yr}_before"], out[f"{m}_{yr}_after"] = b, a
        out[f"{m}_change_{yr}"] = (a / b - 1) if m == "demand_per_day" else (a - b)
    if f"{m}_change_2026" in out and f"{m}_change_2025" in out:
        out[f"{m}_net"] = out[f"{m}_change_2026"] - out[f"{m}_change_2025"]
out = out.sort_values("demand_per_day_2026_before", ascending=False)
out.to_csv(OUT_DIR / "q3_routes_summary.csv")
show = [c for c in out.columns if c.endswith(("_change_2026", "_change_2025", "_net"))]
out[["demand_per_day_2026_before"] + show].round(3).head(30)

## 7. Competitors on PK to UK routes (only if the tables hold other partners)

A new direct carrier or a big competitor fare move around 1 to 3 Sep would show here.

In [ ]:
q4 = f"""
SELECT
  DATE_TRUNC(date, WEEK(SUNDAY)) AS week,
  partner,
  SAFE_DIVIDE(SUM(participation * participation_weight), SUM(participation_weight)) AS participation,
  SAFE_DIVIDE(SUM(pct_of_all_outbound_selections * pct_of_all_outbound_selections_weight),
              SUM(pct_of_all_outbound_selections_weight)) AS share_of_all_selections,
  SUM(participation_weight) AS volume_weight
FROM {TBL_SHOP}
WHERE date BETWEEN '{COMP_FROM}' AND '{END}'
  AND origin IN ({PK})
  AND destination IN ({UK})
GROUP BY week, partner
ORDER BY week, volume_weight DESC
"""
c4 = bq(q4, "q4_competitors_weekly")
c4["week"] = pd.to_datetime(c4["week"])
top = c4.groupby("partner")["volume_weight"].sum().nlargest(8).index
share = c4[c4["partner"].isin(top)].pivot_table(index="week", columns="partner", values="share_of_all_selections")
share.round(3)

## 8. Headline: before vs from 2 Sep, UK+IE vs other, against the same dates in 2025

In [ ]:
def window(df, yr, metric, grp):
    sw = pd.Timestamp(f"{yr}-{SWITCH[5:]}")
    d = df[(df["dest_group"] == grp)]
    before = d[(d["week"] >= sw - pd.Timedelta(days=35)) & (d["week"] < sw - pd.Timedelta(days=6))][metric].mean()
    after  = d[(d["week"] >= sw) & (d["week"] <= sw + pd.Timedelta(days=28))][metric].mean()
    return before, after

rows = []
for df, metric in ((w1, "pct_cheapest"), (w1, "avg_gap_when_more_expensive"), (w1, "participation"),
                   (w1, "demand_per_day"), (w2, "avg_rank"), (w2, "selection_rate_when_present"),
                   (w2, "volume_per_day")):
    for grp in ("UK+IE", "other"):
        for yr in (2025, 2026):
            b, a = window(df, yr, metric, grp)
            rows.append({"metric": metric, "dest_group": grp, "year": yr, "4 weeks before": b, "4 weeks from 2 Sep": a,
                         "change": (a / b - 1) if metric.endswith("per_day") else (a - b)})
headline = pd.DataFrame(rows)
headline.to_csv(OUT_DIR / "headline.csv", index=False)
print("change = relative for volume weights, absolute (points) for rates and rank")
headline.pivot_table(index=["metric", "dest_group"], columns="year", values="change").round(3)

In [ ]:
with pd.ExcelWriter(OUT_DIR / "GFS_PK_UK.xlsx") as xl:
    for name, df in {"coverage": coverage, "user_country": countries, "schema": schema, "partners": partners, "partners_shopping": partners_shop,
                     "competitiveness_weekly": w1, "shopping_weekly": w2, "routes_raw": r3,
                     "routes_summary": out.reset_index(), "competitors_weekly": c4, "headline": headline}.items():
        df.to_excel(xl, sheet_name=name[:31], index=False)
print("Saved", OUT_DIR / "GFS_PK_UK.xlsx")

## How to read it

- **Price**: QR cheapest share down or the gap up on UK+IE but not on other destinations, and not last September,
  means QR became less competitive on Google Flights for PK to UK.
- **Visibility**: participation, top impression share or Best flights down means QR was shown less.
- **Demand**: demand or volume weight down on UK+IE more than on other destinations and more than in 2025 means
  fewer people looked for PK to UK on Google Flights, a market change outside paid search.
- **Nothing moved** on any of these: the drop is on our side (paid search bidding, tracking), not the market.
- Volume weights are relative; trust direction and size of the change, not the level.